In [ ]:
## RAG Application using typesense


In [34]:
import typesense

In [35]:
client = typesense.Client({
    'nodes': [{
        'host': 'k0pnsrc3gxuo1z2vp-1.a2.typesense.net',            
        'port': '443',
        'protocol': 'https'
    }],
    'api_key': 'YOUR_TYPESENSE_API_KEY',
    'connection_timeout_seconds': 2
})  

books_schema = {
    'name': 'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet': True},
        {'name': 'publication_year', 'type': 'int32', 'facet': True},
        {'name': 'ratings_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'},
    ],
    'default_sorting_field': 'ratings_count'
}

try:
    print(client.collections.create(books_schema))
except Exception as e:
    print(f"Collection already exists, skipping creation: {e}")
    print(client.collections['books'].retrieve())

Collection already exists, skipping creation: [Errno 409] A collection with name `books` already exists.
{'created_at': 1770559775, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False,

In [36]:
client

In [37]:
with open('books.jsonl','r',encoding='utf-8') as jsonl_file:
    data=jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [38]:
serach_parameters = {
    'q': 'harry potter',
    'query_by': 'title,authors',
    'filter_by': 'publication_year:>2000',
    'sort_by': 'publication_year:desc'
}
client.collections['books'].documents.search(serach_parameters)

{'facet_counts': [],
 'found': 11,
 'hits': [{'document': {'authors': ['John Tiffany',
     ' Jack Thorne',
     ' J.K. Rowling'],
    'average_rating': 3.75,
    'id': '279',
    'image_url': 'https://images.gr-assets.com/books/1470082995m/29056083.jpg',
    'publication_year': 2016,
    'ratings_count': 270603,
    'title': 'Harry Potter and the Cursed Child, Parts One and Two'},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Cursed Child, Parts One and Two'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': '<mark>Harry</mark> <mark>Potter</mark> and the Cursed Child, Parts One and Two'}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0

In [39]:
serach_parameters = {
    'q': 'harry potter',
    'query_by': 'title',
    'facet_by': 'authors',
    'sort_by': 'average_rating:desc'
}
client.collections['books'].documents.search(serach_parameters)

{'facet_counts': [{'counts': [{'count': 10,
     'highlighted': 'J.K. Rowling',
     'value': 'J.K. Rowling'},
    {'count': 8, 'highlighted': ' Mary GrandPré', 'value': ' Mary GrandPré'},
    {'count': 2, 'highlighted': ' J.K. Rowling', 'value': ' J.K. Rowling'},
    {'count': 1, 'highlighted': 'Bob McCabe', 'value': 'Bob McCabe'},
    {'count': 1, 'highlighted': 'Brian Sibley', 'value': 'Brian Sibley'},
    {'count': 1, 'highlighted': 'David Colbert', 'value': 'David Colbert'},
    {'count': 1, 'highlighted': 'David Baggett', 'value': 'David Baggett'},
    {'count': 1, 'highlighted': 'Melissa Anelli', 'value': 'Melissa Anelli'},
    {'count': 1, 'highlighted': 'John   Williams', 'value': 'John   Williams'},
    {'count': 1,
     'highlighted': ' Shawn E. Klein',
     'value': ' Shawn E. Klein'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 13}}],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling'],
    'average_rating': 4.74,
    'id': 

In [40]:
### langchain +typesense +Groq LLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [41]:
import os
os.environ['GROQ_API_KEY'] = "YOUR_GROQ_API_KEY"

In [42]:
loader = TextLoader('test.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(documents)    

embeddings = HuggingFaceEmbeddings()


C:\Users\darsh\AppData\Local\Temp\ipykernel_33176\1084843016.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 444.19it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
docsearch = Typesense.from_documents(texts, embeddings, 
                                     typesense_client_params={
                                         "host": "k0pnsrc3gxuo1z2vp-1.a2.typesense.net",
                                            "port": "443",
                                            "protocol": "https",
                                            "typesense_api_key": "YOUR_TYPESENSE_API_KEY",
                                            "typesense_collection_name": "langchain_docs"
                                     })

In [44]:
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x000002353C7887D0>, search_kwargs={})

In [45]:
query = "What is artificial intelligence?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence: A Comprehensive Overview

Artificial Intelligence (AI) is one of the most transformative technologies of the modern era. It refers to the simulation of human intelligence processes by computer systems. These processes include learning, reasoning, problem-solving, perception, and language understanding. AI has evolved from a theoretical concept to a practical technology that powers countless applications across industries worldwide.

The history of artificial intelligence dates back to the 1950s when pioneers like Alan Turing, John McCarthy, and Marvin Minsky laid the groundwork for this field. Alan Turing proposed the famous Turing Test, which evaluates a machine's ability to exhibit intelligent behavior indistinguishable from that of a human. John McCarthy coined the term "Artificial Intelligence" in 1956 at the Dartmouth Conference, which is widely considered the birth of AI as a formal academic discipline.


In [46]:
query = "What is artificial intelligence?"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content='Artificial Intelligence: A Comprehensive Overview\n\nArtificial Intelligence (AI) is one of the most transformative technologies of the modern era. It refers to the simulation of human intelligence processes by computer systems. These processes include learning, reasoning, problem-solving, perception, and language understanding. AI has evolved from a theoretical concept to a practical technology that powers countless applications across industries worldwide.\n\nThe history of artificial intelligence dates back to the 1950s when pioneers like Alan Turing, John McCarthy, and Marvin Minsky laid the groundwork for this field. Alan Turing proposed the famous Turing Test, which evaluates a machine\'s ability to exhibit intelligent behavior indistinguishable from that of a human. John McCarthy coined the term "Artificial Intelligence" in 1956 at the Dartmouth Conference, which is widely considered the birth of AI as a formal academic dis